<a href="https://colab.research.google.com/github/Karsuman4298/Machine_learning/blob/main/RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import shuffle
import matplotlib.pyplot as pyplot
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import torch

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


In [ ]:
df=pd.read_csv('new.csv')

In [ ]:
df.head(5)

In [ ]:
target_col=df.columns[-1]
X=df.drop(target_col,axis=1)
y=df[target_col]
le=LabelEncoder()
y=le.fit_transform(y)
X=pd.get_dummies(X)

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
overall_mask = pd.Series(True, index=X.index)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR


    overall_mask = overall_mask & ((X[col] >= lower) & (X[col] <= upper))

X = X[overall_mask]
y = y[overall_mask]

print(f"Shape of X after outlier removal: {X.shape}")
print(f"Shape of y after outlier removal: {y.shape}")


In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
!pip install scikit-optimize

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=0)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

best_model.fit(X_train_res, y_train_res)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Count classes
before_counts = pd.Series(y_train).value_counts()

print("Before SMOTE:")
print(before_counts)

# Plot
before_counts.plot(kind='bar')
plt.title("Class Distribution Before SMOTE")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=0)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [ ]:
after_counts = pd.Series(y_train_smote).value_counts()

print("After SMOTE:")
print(after_counts)

# Plot
after_counts.plot(kind='bar')
plt.title("Class Distribution After SMOTE")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,4))

before_counts.plot(kind='bar', ax=ax[0])
ax[0].set_title("Before SMOTE")

after_counts.plot(kind='bar', ax=ax[1])
ax[1].set_title("After SMOTE")

plt.show()

In [ ]:
from imblearn.pipeline import Pipeline
pipeline = Pipeline([
    ('smote', SMOTE(random_state=0)),
    ('rf', RandomForestClassifier(random_state=0, n_jobs=-1))
])

# Hyperparameter search space
search_space = {
    'rf__n_estimators': (100, 600),
    'rf__max_depth': (5, 50),
    'rf__min_samples_split': (2, 20),
    'rf__min_samples_leaf': (1, 10),
    'rf__max_features': ['sqrt', 'log2', None]
}



In [ ]:
opt = BayesSearchCV(
    pipeline,
    search_space,
    n_iter=40,
    cv=5,
    scoring='f1',
    random_state=0,
    n_jobs=-1
)


opt.fit(X_train, y_train)

best_model = opt.best_estimator_
print(best_model)

In [ ]:

best_model = opt.best_estimator_

y_pred = best_model.predict(X_test)

print("Best Parameters:", opt.best_params_)
print(classification_report(y_test, y_pred))